# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing a Croissant-schematized dataset using the `mlcroissant` library.

### Dataset Source
The dataset is sourced from a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not index or iterate over dataset.metadata)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets and their IDs
print('Record Sets:')
record_sets_info = {}
for rs in dataset.record_sets:
    print(f"- @id: {rs['@id']}, Name: {rs.get('name', '(no name)')}")
    record_sets_info[rs['@id']] = rs

# For each record set, list fields and their IDs
for rs_id, rs in record_sets_info.items():
    print(f"\nFields in Record Set {rs_id}:")
    fields = rs.get('field', [])
    # field can be a list or single dict
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  - @id: {field['@id']}, Name: {field.get('name', '(no name)')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose the main record set for actual records
# Let's find the record set that likely holds survey responses
# For demonstration, choose the first record set

main_record_set_id = None
if dataset.record_sets:
    main_record_set_id = dataset.record_sets[0]['@id']

# Compile a list of all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Show available columns from the main record set (referenced by @id)
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns in record set {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No data found for main record set.")

## 4. Exploratory Data Analysis (EDA)
Filter records and normalize numeric fields. Group by categorical fields.

In [ ]:
# Choose numeric and group field by @id
# Inspect columns - e.g. 'phq9_score', 'age', 'gad7_score', etc.
survey_df = dataframes.get(main_record_set_id, pd.DataFrame())

numeric_field = None
possible_numeric_fields = []
for col in survey_df.columns:
    # Pick a likely numeric field (example: PHQ-9 score, GAD-7 score, Age)
    if 'score' in col or 'age' in col:
        possible_numeric_fields.append(col)
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # e.g. 'phq9_score'

# Choose a group field (e.g. 'gender', 'village', 'level_of_education')
group_field = None
possible_group_fields = [c for c in survey_df.columns if any(g in c.lower() for g in ['gender', 'village', 'education', 'marital'])]
if possible_group_fields:
    group_field = possible_group_fields[0]

if numeric_field:
    # Filter records based on a threshold
    threshold = 10
    filtered_df = survey_df[survey_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by group_field if present
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and relationships between group and numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field histogram
if numeric_field and not survey_df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(survey_df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group field is available, plot group means
    if group_field and group_field in survey_df.columns:
        plt.figure(figsize=(8,5))
        group_means = survey_df.groupby(group_field)[numeric_field].mean().sort_values()
        sns.barplot(x=group_means.values, y=group_means.index)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
This notebook has:
- Loaded metadata and records from a Croissant-schematized dataset using `mlcroissant`.
- Reviewed available record sets, fields, and their unique `@id`s.
- Extracted survey data and performed exploratory analysis based on numeric and grouping fields referenced by their `@id`s.
- Visualized the distribution of mental health scores and explored demographic differences.

Key insights from the EDA may be used to inform further health analysis, policy, or model development for the Kilifi County dataset.
